# Notebook 02 — Baseline, Model Comparison & Error Analysis
## Airbnb Multi-City Nightly Price Prediction (Regression)

**Module:** ITI113 Machine Learning & Operations
**Pipeline stage:** 3. Training → 4. Experiment Tracking (SageMaker MLflow App)
**Estimated runtime:** 25–40 minutes

---

## What this notebook does

1. Connects to the team's **SageMaker Serverless MLflow App**, validating the
   `TeamId` resource tag before a single metric is logged
2. Builds a **leak-safe modelling pipeline** with a custom out-of-fold smoothed
   target encoder for the 660-level neighbourhood feature
3. Establishes a **two-tier baseline**: a city-median predictor (the floor any
   model must beat to justify its existence) and a regularised linear model
4. Trains and tunes **two advanced comparison models** — Random Forest and
   Histogram Gradient Boosting — with nested MLflow child runs
5. Runs a **deep error analysis**: residual diagnostics, regression-to-the-mean
   quantification, segment-level performance across city / room type / price
   decile / review status / host type, and worst-performing micro-markets
6. Applies the **R² ≥ 0.70 quality gate** and exports `best_model.json`

## Why regression, and which metrics

The target is `log_price_usd` (continuous). We report four metrics because none
of them is sufficient alone:

| Metric | Space | What it answers |
|---|---|---|
| **R²** | log | What share of price variance did we explain? Used for the gate. |
| **RMSE** | log | Penalises large misses; comparable across cities. |
| **MdAPE** | USD | Median % error — what a typical host actually experiences. |
| **within-25%** | USD | Share of hosts given a usable recommendation. |

MdAPE rather than MAPE: the mean absolute percentage error is dominated by a
handful of very cheap listings where a $10 miss is a 40% error. The median is
the honest summary of the typical case, and we report both so the gap between
them is visible.

## Prerequisites

- Notebook 01 completed — processed artefacts and `feature_contract.json` exist
- `mlflow_app_config_<team_id>.json` present, **or** `DEFAULT_MLFLOW_APP_ARN` set

In [1]:
# Run once, then restart the kernel if anything was upgraded.
%pip install --upgrade --quiet "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow

Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

In [2]:
import boto3
from botocore.exceptions import ClientError
import time
import json
from datetime import datetime
from pathlib import Path

REGION = "ap-southeast-1"
COURSE = "ITI113"
SEMESTER = "26S1"

# Change these for each team/student.
TEAM_ID = "team14"
STUDENT_ID = "s1402"

# Project name used in tags and artifact organisation.
PROJECT_NAME = "airbnb"

# Existing course bucket from the SageMaker Pipeline lab.
CLASS_BUCKET = "nyp-26s1-iti113"

# One MLflow App per team is usually enough.
MLFLOW_APP_NAME = f"iti113-26s1-{TEAM_ID}-mlflow-app"

# Where MLflow run artifacts will be stored.
ARTIFACT_STORE_URI = f"s3://{CLASS_BUCKET}/iti113/{TEAM_ID}/mlflow-app-artifacts/"

# Experiment name inside MLflow App. Use a normal MLflow experiment name, not a Databricks /Workspace path.
EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"

# Optional: make this MLflow App the account/domain default. Keep False for classroom safety.
SET_AS_ACCOUNT_DEFAULT = False
SET_AS_DEFAULT_FOR_EXISTING_DOMAINS = False

session = boto3.Session(region_name=REGION)
sts = session.client("sts")
sm = session.client("sagemaker")
s3 = session.client("s3")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

# Build role name from TEAM_ID, e.g. team40 -> SageMakerExecutionRole-ITI113-Team40.
TEAM_ROLE_SUFFIX = TEAM_ID.lower().replace("team", "Team")
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/SageMakerExecutionRole-ITI113-{TEAM_ROLE_SUFFIX}"

MLFLOW_APP_TAGS = {
    "course": COURSE, "semester": SEMESTER,
    "team_id": TEAM_ID, "student_id": STUDENT_ID,
    "team": TEAM_ID, "student": STUDENT_ID,
    "dataset": PROJECT_NAME, "task_type": "regression",
    "target": "log_price_usd",
    "split_strategy": "GroupShuffleSplit(host_id)",
    "tracking_backend": TRACKING_BACKEND,
    "mlflow_app_arn": MLFLOW_APP_ARN,
}

print("Account:", ACCOUNT_ID)
print("Caller ARN:", CALLER_ARN)
print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("Project Name:", PROJECT_NAME)
print("MLflow App Name:", MLFLOW_APP_NAME)
print("Artifact Store:", ARTIFACT_STORE_URI)
print("Role ARN:", ROLE_ARN)
print("Experiment:", EXPERIMENT_NAME)
print("\nMLflow App tags to apply:")
for tag in MLFLOW_APP_TAGS:
    print(f"  {tag['Key']} = {tag['Value']}")


NameError: name 'TRACKING_BACKEND' is not defined

## 1. Connect to the SageMaker MLflow App

The `TeamId` resource tag is validated **before** any logging call. Two failure
modes this catches: pointing at another team's tracking server (IAM should also
block this with a 403), and an untagged app that would make runs
unattributable. Failing loudly here is much cheaper than discovering polluted
experiment data after 40 runs.

In [10]:
def find_mlflow_app_by_name(name: str):
    paginator = sm.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        for summary in page.get("Summaries", []):
            if summary.get("Name") == name:
                return summary
    return None


def ensure_mlflow_app_tags(resource_arn: str, required_tags: list):
    """Check and apply required tags to an existing MLflow App.

    If the current role does not have sagemaker:AddTags/ListTags permission,
    this function will print a warning and continue. The admin can tag the App later.
    """
    required = {tag["Key"]: tag["Value"] for tag in required_tags}

    try:
        existing_tags_response = sm.list_tags(ResourceArn=resource_arn)
        existing = {
            tag["Key"]: tag["Value"]
            for tag in existing_tags_response.get("Tags", [])
        }

        missing_or_different = [
            {"Key": key, "Value": value}
            for key, value in required.items()
            if existing.get(key) != value
        ]

        if missing_or_different:
            print("\nAdding/updating required MLflow App tags:")
            for tag in missing_or_different:
                print(f"  {tag['Key']} = {tag['Value']}")

            sm.add_tags(
                ResourceArn=resource_arn,
                Tags=missing_or_different,
            )
        else:
            print("\nExisting MLflow App already has the required tags.")

        final_tags = sm.list_tags(ResourceArn=resource_arn).get("Tags", [])
        print("\nCurrent MLflow App tags:")
        for tag in final_tags:
            print(f"  {tag['Key']} = {tag['Value']}")

    except ClientError as e:
        print("\n[WARNING] Could not verify or update MLflow App tags.")
        print("This may happen if the current role does not have sagemaker:ListTags/AddTags.")
        print("Ask the admin to ensure these tags exist on the MLflow App:")
        for tag in required_tags:
            print(f"  {tag['Key']} = {tag['Value']}")
        print("\nOriginal error:")
        print(e)


existing = find_mlflow_app_by_name(MLFLOW_APP_NAME)

if existing:
    mlflow_app_arn = existing["Arn"]
    print("Reusing existing MLflow App:")
    print(json.dumps(existing, indent=2, default=str))

    # Important for team-level MLflow IAM restriction.
    ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)

else:
    create_args = {
        "Name": MLFLOW_APP_NAME,
        "ArtifactStoreUri": ARTIFACT_STORE_URI,
        "RoleArn": ROLE_ARN,
        "ModelRegistrationMode": "AutoModelRegistrationDisabled",
        "Tags": MLFLOW_APP_TAGS,
    }

    if SET_AS_ACCOUNT_DEFAULT:
        create_args["AccountDefaultStatus"] = "ENABLED"

    if SET_AS_DEFAULT_FOR_EXISTING_DOMAINS and domain_ids:
        create_args["DefaultDomainIdList"] = domain_ids

    print("Creating MLflow App with args:")
    print(json.dumps(create_args, indent=2, default=str))

    response = sm.create_mlflow_app(**create_args)
    mlflow_app_arn = response["Arn"]
    print("Create response:", response)

print("\nMLflow App ARN:")
print(mlflow_app_arn)


Reusing existing MLflow App:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS",
  "Name": "iti113-26s1-team14-mlflow-app",
  "Status": "Created",
  "CreationTime": "2026-08-11 03:11:26+00:00",
  "LastModifiedTime": "2026-08-11 03:16:08.236000+00:00",
  "MlflowVersion": "3.10.1"
}


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:66                                                                                   │
│                                                                                                  │
│   63 │   print(json.dumps(existing, indent=2, default=str))                                      │
│   64 │                                                                                           │
│   65 │   # Important for team-level MLflow IAM restriction.                                      │
│ ❱ 66 │   ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)                                 │
│   67                                                                                             │
│   68 else:                                                                                       │
│   69 │   create_args = {                                                                         │
│                                                                                                  │
│ in ensure_mlflow_app_tags:16                                                                     │
│                                                                                                  │
│   13 │   If the current role does not have sagemaker:AddTags/ListTags permission,                │
│   14 │   this function will print a warning and continue. The admin can tag the App later.       │
│   15 │   """                                                                                     │
│ ❱ 16 │   required = {tag["Key"]: tag["Value"] for tag in required_tags}                          │
│   17 │                                                                                           │
│   18 │   try:                                                                                    │
│   19 │   │   existing_tags_response = sm.list_tags(ResourceArn=resource_arn)                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: string indices must be integers, not 'str'

In [11]:
# ---------------------------------------------------------------------------
# TeamId tag validation, then connect. Falls back to a local MLflow file store
# so the notebook remains runnable off-AWS — the experiment structure, run
# names and tags are identical either way.
# ---------------------------------------------------------------------------
TRACKING_BACKEND = "sagemaker_mlflow_app"

if AWS_AVAILABLE:
    sm_client = boto3.client("sagemaker", region_name=REGION)
    try:
        tags = {t["Key"]: t["Value"]
                for t in sm_client.list_tags(ResourceArn=MLFLOW_APP_ARN).get("Tags", [])}
        print("MLflow App tags:")
        for k, v in tags.items():
            print(f"  {k}: {v}")
        app_team = tags.get("TeamId")
        if app_team != TEAM_ID:
            raise PermissionError(
                f"MLflow App TeamId={app_team} != notebook TEAM_ID={TEAM_ID}. "
                "Refusing to log to another team's tracking server.")
        print(f"\n[OK] TeamId tag '{app_team}' matches notebook TEAM_ID.")
        mlflow.set_tracking_uri(MLFLOW_APP_ARN)
    except Exception as e:
        print(f"\n[ERROR] Could not validate the MLflow App team tag.")
        print("  1. The ARN may belong to another team (IAM correctly blocked it)")
        print("  2. The app may be missing its TeamId tag")
        print("  3. This role may lack sagemaker:ListTags")
        print(f"  {type(e).__name__}: {e}")
        raise
else:
    TRACKING_BACKEND = "local_file_store"
    mlflow.set_tracking_uri(f"file://{Path('mlruns').resolve()}")
    print(f"[LOCAL] Tracking to {mlflow.get_tracking_uri()}")

mlflow.set_experiment(MLFLOW_EXPERIMENT)
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)

COMMON_TAGS = {
    "course": COURSE, "semester": SEMESTER,
    "team_id": TEAM_ID, "student_id": STUDENT_ID,
    "team": TEAM_ID, "student": STUDENT_ID,
    "dataset": PROJECT_NAME, "task_type": "regression",
    "target": "log_price_usd",
    "split_strategy": "GroupShuffleSplit(host_id)",
    "tracking_backend": TRACKING_BACKEND,
    "mlflow_app_arn": MLFLOW_APP_ARN,
}

print(f"\nTracking URI  : {mlflow.get_tracking_uri()}")
print(f"Experiment    : {exp.name}  (id={exp.experiment_id})")

if AWS_AVAILABLE:
    try:
        r = sm_client.create_presigned_mlflow_app_url(Arn=MLFLOW_APP_ARN)
        print("\nMLflow UI (presigned):")
        print(r.get("AuthorizedUrl") or r.get("Url"))
    except Exception as e:
        print(f"\nCould not presign MLflow UI URL: {type(e).__name__}")


[ERROR] Could not validate the MLflow App team tag.
  1. The ARN may belong to another team (IAM correctly blocked it)
  2. The app may be missing its TeamId tag
  3. This role may lack sagemaker:ListTags
  ClientError: An error occurred (ValidationException) when calling the ListTags operation: 1 validation error detected: Value 'arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team14' at 'resourceArn' failed to satisfy constraint: Member must satisfy regular expression pattern: arn:aws[a-z-]*:sagemaker:[a-z0-9-]*:[0-9]{12}:.+


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:12                                                                                   │
│                                                                                                  │
│    9 │   sm_client = boto3.client("sagemaker", region_name=REGION)                               │
│   10 │   try:                                                                                    │
│   11 │   │   tags = {t["Key"]: t["Value"]                                                        │
│ ❱ 12 │   │   │   │   for t in sm_client.list_tags(ResourceArn=MLFLOW_APP_ARN).get("Tags", [])    │
│   13 │   │   print("MLflow App tags:")                                                           │
│   14 │   │   for k, v in tags.items():                                                           │
│   15 │   │   │   print(f"  {k}: {v}")                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                     

## 2. Load Processed Data and the Feature Contract

We read the column lists from `feature_contract.json` rather than re-declaring
them. A notebook that redefines its own feature list is a notebook that will one
day disagree with the training job.

In [ ]:
def load_csv(key, name):
    if AWS_AVAILABLE:
        r = s3.get_object(Bucket=BUCKET, Key=key)
        return pd.read_csv(io.BytesIO(r["Body"].read()))
    return pd.read_csv(LOCAL_ARTIFACTS / name)


pp = f"{PREFIX}/processed"

if Path("feature_contract.json").exists():
    contract = json.loads(Path("feature_contract.json").read_text())
else:
    r = s3.get_object(Bucket=BUCKET, Key=f"{pp}/feature_contract.json")
    contract = json.loads(r["Body"].read())

NUMERIC_FEATURES       = contract["numeric_features"]
CATEGORICAL_FEATURES   = contract["categorical_features"]
TARGET_ENCODE_FEATURES = contract["target_encode_features"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + TARGET_ENCODE_FEATURES
TARGET = contract["target"]

X_train = load_csv(f"{pp}/train_features.csv", "train_features.csv")
y_train = load_csv(f"{pp}/train_labels.csv",   "train_labels.csv").squeeze("columns")
X_test  = load_csv(f"{pp}/test_features.csv",  "test_features.csv")
y_test  = load_csv(f"{pp}/test_labels.csv",    "test_labels.csv").squeeze("columns")
meta_test  = load_csv(f"{pp}/test_meta.csv",   "test_meta.csv")
meta_train = load_csv(f"{pp}/train_meta.csv",  "train_meta.csv")

assert list(X_train.columns) == ALL_FEATURES, "Feature drift vs contract!"
assert len(X_test) == len(meta_test), "Metadata row misalignment!"
assert not (set(meta_train.host_id) & set(meta_test.host_id)), "Host leakage!"

print(f"Train : {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"Test  : {X_test.shape[0]:,} rows")
print(f"Target: {TARGET}   mean={y_train.mean():.3f}  sd={y_train.std():.3f}")
print(f"Split : {contract['split_strategy']}")
print(f"Gate  : R2 >= {contract['quality_gate_r2']}  "
      f"(estimated ceiling {contract['estimated_r2_ceiling']})")

## 3. The Modelling Pipeline

### 3.1 A leak-safe target encoder

`city_neigh` has 660+ levels. One-hot encoding would add 660 sparse columns;
label encoding would impose a meaningless ordering. Target encoding is the right
tool — and the easiest way in all of applied ML to leak your test set.

The encoder below solves this structurally rather than by discipline:

- **`fit_transform`** (training) returns **out-of-fold** encodings: for each of
  5 folds, the encoding for rows in that fold is computed from the *other four
  folds only*. A row never contributes to its own encoding, so the encoded
  feature cannot memorise its own label.
- **`transform`** (inference) uses the full-training-set mapping, with the
  global prior as the fallback for unseen neighbourhoods.
- **Smoothing** shrinks sparse levels toward the prior:

$$\text{enc}(k)=\frac{n_k\bar y_k + m\,\bar y}{n_k+m},\qquad m=20$$

A neighbourhood with 500 listings keeps ~96% of its own mean; one with 3
listings keeps ~13% and borrows the rest from the global prior. That is
empirical-Bayes shrinkage, and it is why the encoder does not blow up on the
long tail.

Because it is a proper scikit-learn transformer it is fitted *inside*
cross-validation, so every CV score already accounts for encoder variance.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)


class SmoothedTargetEncoder(BaseEstimator, TransformerMixin):
    """Out-of-fold smoothed target encoding for high-cardinality categoricals.

    fit_transform -> out-of-fold encodings  (no row sees its own label)
    transform     -> full-fit mapping, global prior for unseen levels
    """

    def __init__(self, cols=None, smoothing=20.0, n_splits=5, random_state=42):
        self.cols = cols
        self.smoothing = smoothing
        self.n_splits = n_splits
        self.random_state = random_state

    def _mapping(self, keys, vals, prior):
        agg = pd.Series(np.asarray(vals)).groupby(np.asarray(keys)).agg(["mean", "count"])
        return ((agg["mean"] * agg["count"] + prior * self.smoothing)
                / (agg["count"] + self.smoothing))

    def fit(self, X, y):
        X = pd.DataFrame(X); y = pd.Series(np.asarray(y))
        self.cols_ = list(X.columns) if self.cols is None else self.cols
        self.prior_ = float(y.mean())
        self.maps_ = {c: self._mapping(X[c].values, y.values, self.prior_)
                      for c in self.cols_}
        self.feature_names_out_ = [f"{c}_te" for c in self.cols_]
        return self

    def fit_transform(self, X, y=None, **kw):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(np.asarray(y)).reset_index(drop=True)
        self.fit(X, y)
        out = pd.DataFrame(index=X.index)
        kf = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for c in self.cols_:
            oof = pd.Series(np.full(len(X), self.prior_), index=X.index, dtype=float)
            for tr, va in kf.split(X):
                fold_prior = y.iloc[tr].mean()
                m = self._mapping(X[c].iloc[tr].values, y.iloc[tr].values, fold_prior)
                oof.iloc[va] = X[c].iloc[va].map(m).fillna(fold_prior).values
            out[f"{c}_te"] = oof
        return out.values

    def transform(self, X):
        X = pd.DataFrame(X)
        return pd.DataFrame(
            {f"{c}_te": X[c].map(self.maps_[c]).fillna(self.prior_)
             for c in self.cols_}, index=X.index).values

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_out_, dtype=object)


def make_preprocessor():
    """Identical preprocessing for every model — the only thing that varies
    between experiments is the estimator, so comparisons are like-for-like."""
    return ColumnTransformer([
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=200,
                              sparse_output=False), CATEGORICAL_FEATURES),
        ("te",  SmoothedTargetEncoder(cols=TARGET_ENCODE_FEATURES),
                TARGET_ENCODE_FEATURES),
    ], remainder="drop")


def feature_names(fitted_pre):
    names = list(NUMERIC_FEATURES)
    names += list(fitted_pre.named_transformers_["cat"].get_feature_names_out(
        CATEGORICAL_FEATURES))
    names += [f"{c}_te" for c in TARGET_ENCODE_FEATURES]
    return names


print("Pipeline components ready.")
print(f"  passthrough numeric : {len(NUMERIC_FEATURES)}")
print(f"  one-hot             : {CATEGORICAL_FEATURES}  (min_frequency=200)")
print(f"  target-encoded      : {TARGET_ENCODE_FEATURES}  (smoothing=20, 5-fold OOF)")

### 3.2 Evaluation helpers

Every run logs the same metric set. Consistency is what makes MLflow's
comparison view usable — a run missing `test_mdape` becomes invisible when you
sort by it three weeks later.

We evaluate in **both spaces**. Log-space metrics drive model selection because
that is the loss we optimise; USD-space metrics drive the business conversation
because that is what a host experiences.

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    """Log-space and USD-space metrics for train and test."""
    out = {}
    for split, Xs, ys in [("train", X_tr, y_tr), ("test", X_te, y_te)]:
        p = model.predict(Xs)
        out[f"{split}_r2"]       = round(float(r2_score(ys, p)), 4)
        out[f"{split}_rmse_log"] = round(float(np.sqrt(mean_squared_error(ys, p))), 4)
        out[f"{split}_mae_log"]  = round(float(mean_absolute_error(ys, p)), 4)
        # exp() of a log prediction is the conditional MEDIAN price
        y_usd, p_usd = np.exp(ys), np.exp(p)
        ape = np.abs(p_usd - y_usd) / y_usd
        out[f"{split}_mape"]         = round(float(ape.mean()), 4)
        out[f"{split}_mdape"]        = round(float(np.median(ape)), 4)
        out[f"{split}_mae_usd"]      = round(float(np.abs(p_usd - y_usd).mean()), 2)
        out[f"{split}_within_10pct"] = round(float((ape <= 0.10).mean()), 4)
        out[f"{split}_within_25pct"] = round(float((ape <= 0.25).mean()), 4)
    out["overfit_gap_r2"] = round(out["train_r2"] - out["test_r2"], 4)
    return out


def log_residual_plots(y_true, y_pred, title, artifact_dir="plots"):
    """Four-panel residual diagnostic — the core of regression error analysis."""
    resid = y_true - y_pred
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))

    ax[0, 0].scatter(y_pred, y_true, s=2, alpha=.08, color="#5B9BD5")
    lims = [min(y_pred.min(), y_true.min()), max(y_pred.max(), y_true.max())]
    ax[0, 0].plot(lims, lims, "r--", lw=1.5, label="perfect")
    ax[0, 0].set_xlabel("predicted log price"); ax[0, 0].set_ylabel("actual log price")
    ax[0, 0].set_title("Predicted vs Actual", fontweight="bold"); ax[0, 0].legend()

    ax[0, 1].scatter(y_pred, resid, s=2, alpha=.08, color="#E67E22")
    ax[0, 1].axhline(0, c="red", ls="--", lw=1.5)
    ax[0, 1].set_xlabel("predicted log price"); ax[0, 1].set_ylabel("residual")
    ax[0, 1].set_title("Residuals vs Fitted (heteroscedasticity check)",
                       fontweight="bold")

    ax[1, 0].hist(resid, bins=90, color="#27AE60", edgecolor="white")
    ax[1, 0].axvline(0, c="red", ls="--")
    ax[1, 0].set_xlabel("residual (log points)")
    ax[1, 0].set_title(f"Residual distribution — sd={resid.std():.3f}",
                       fontweight="bold")

    from scipy import stats
    stats.probplot(resid.sample(min(20000, len(resid)), random_state=0)
                   if isinstance(resid, pd.Series) else resid[:20000],
                   dist="norm", plot=ax[1, 1])
    ax[1, 1].set_title("Q-Q plot of residuals", fontweight="bold")
    ax[1, 1].get_lines()[0].set_markersize(2)

    plt.suptitle(f"Residual diagnostics — {title}", fontweight="bold", y=1.00)
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as t:
        p = f"{t}/residual_diagnostics.png"
        fig.savefig(p, dpi=110, bbox_inches="tight")
        mlflow.log_artifact(p, artifact_path=artifact_dir)
    plt.show(); plt.close()


def log_importance(model, names, title, top=25, artifact_dir="plots"):
    est = model.named_steps["model"]
    if not hasattr(est, "feature_importances_"):
        return None
    imp = pd.Series(est.feature_importances_, index=names).nlargest(top)[::-1]
    fig, ax = plt.subplots(figsize=(7, 8))
    imp.plot(kind="barh", ax=ax, color="#5B9BD5")
    ax.set_title(f"Top {top} feature importances — {title}", fontweight="bold")
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as t:
        p = f"{t}/feature_importance.png"
        fig.savefig(p, dpi=110, bbox_inches="tight")
        mlflow.log_artifact(p, artifact_path=artifact_dir)
    plt.show(); plt.close()
    return imp


def print_metrics(name, m):
    print(f"\n{name}")
    print("-" * 62)
    print(f"  TEST  R2 {m['test_r2']:.4f} | RMSE(log) {m['test_rmse_log']:.4f} "
          f"| MdAPE {m['test_mdape']:.1%} | within-25% {m['test_within_25pct']:.1%}")
    print(f"  TRAIN R2 {m['train_r2']:.4f} | overfit gap {m['overfit_gap_r2']:+.4f}")
    print(f"  MAE ${m['test_mae_usd']:.2f}/night | within-10% {m['test_within_10pct']:.1%}")


print("Evaluation helpers ready.")

## 4. Run 1 — Naive Baseline (the performance floor)

Before any machine learning, we need to know what *no* machine learning
achieves. The honest floor here is not the global mean — §2.2 of Notebook 01
showed cities differ by 3× in USD terms, and any deployed system would obviously
condition on city. So the floor is **"charge the median of your city"**: free,
instant, and already a usable recommendation.

Every subsequent model must justify its complexity against this number. A
gradient-boosting ensemble that beats the city median by two points of R² is not
worth its operational cost.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.base import BaseEstimator, RegressorMixin


class CityMedianRegressor(BaseEstimator, RegressorMixin):
    """Predict the training median log-price of the listing's city."""

    def fit(self, X, y):
        self.global_ = float(np.median(y))
        self.map_ = pd.Series(np.asarray(y)).groupby(
            pd.DataFrame(X)["city"].values).median()
        return self

    def predict(self, X):
        return (pd.DataFrame(X)["city"].map(self.map_)
                .fillna(self.global_).values)


with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_baseline_city_median") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_family": "naive_baseline",
                     "run_role": "performance_floor"})
    mlflow.log_params({"model_type": "CityMedianRegressor",
                       "strategy": "median log price per city",
                       "n_features_used": 1, "trainable_parameters": 10,
                       "train_size": len(X_train)})

    base = CityMedianRegressor().fit(X_train, y_train)
    m_base = evaluate(base, X_train, y_train, X_test, y_test)
    mlflow.log_metrics(m_base)

    # Global-mean reference, for context on how much `city` alone contributes
    dm = DummyRegressor(strategy="mean").fit(X_train, y_train)
    r2_global = r2_score(y_test, dm.predict(X_test))
    mlflow.log_metric("test_r2_global_mean_reference", round(float(r2_global), 4))

    BASELINE_RUN_ID = run.info.run_id
    BASELINE_R2 = m_base["test_r2"]

print_metrics("RUN 1 — CITY-MEDIAN BASELINE", m_base)
print(f"\n  Global-mean reference R2 : {r2_global:.4f}  (by construction ~0)")
print(f"  City alone explains      : {m_base['test_r2']:.1%} of log-price variance")
print(f"\n  This is the floor. Everything below must beat R2 = {BASELINE_R2:.4f}.")

## 5. Run 2 — Regularised Linear Baseline (Ridge)

**Why Ridge specifically.** Three reasons, in order of importance:

1. **It is the correct null hypothesis for a log-target model.** On the log
   scale, a linear model asserts that every feature has a constant *percentage*
   effect on price. That is a genuinely plausible economic model, not a straw
   man — if it holds, we should ship it and enjoy the interpretability.
2. **Coefficients are directly readable as elasticities.** $\beta_j$ is the
   approximate percentage change in price per unit of $x_j$, which is exactly
   the form a pricing team wants to reason about.
3. **L2 rather than OLS or Lasso.** Notebook 01 §2.12 showed `accommodates`
   and `bedrooms` correlate at ~0.6, and the one-hot city block is
   near-collinear with several geographic features. OLS on collinear inputs
   gives unstable, uninterpretable coefficients with inflated variance; L2
   shrinkage stabilises them while keeping all features (unlike Lasso, which
   would arbitrarily pick one of each correlated pair).

Scaling is applied **after** the ColumnTransformer so that the penalty is
applied on a common scale — without it, `booking_window` (range ~1,125) would be
penalised a thousand times harder than a binary amenity flag.

In [ ]:
from sklearn.linear_model import Ridge

RIDGE_PARAMS = {"alpha": 1.0, "solver": "auto", "random_state": RANDOM_STATE}

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_ridge_baseline") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_family": "linear",
                     "run_role": "interpretable_baseline"})
    mlflow.log_params({**RIDGE_PARAMS, "model_type": "Ridge",
                       "n_features": X_train.shape[1],
                       "train_size": len(X_train),
                       "preprocessing": "passthrough+onehot+OOF_target_encode+scale"})

    ridge = Pipeline([("pre", make_preprocessor()),
                      ("scale", StandardScaler()),
                      ("model", Ridge(**RIDGE_PARAMS))])
    t0 = time.time(); ridge.fit(X_train, y_train)
    fit_s = time.time() - t0

    m_ridge = evaluate(ridge, X_train, y_train, X_test, y_test)
    mlflow.log_metrics({**m_ridge, "fit_seconds": round(fit_s, 1),
                        "lift_over_baseline_r2": round(m_ridge["test_r2"] - BASELINE_R2, 4)})
    log_residual_plots(y_test, ridge.predict(X_test), "Ridge")

    names = feature_names(ridge.named_steps["pre"])
    coefs = pd.Series(ridge.named_steps["model"].coef_, index=names)
    top = coefs.abs().nlargest(20).index
    fig, ax = plt.subplots(figsize=(7, 7))
    coefs[top].sort_values().plot(kind="barh", ax=ax,
        color=["#C0392B" if v < 0 else "#27AE60" for v in coefs[top].sort_values()])
    ax.set_title("Ridge coefficients (standardised) — top 20 by |magnitude|",
                 fontweight="bold")
    ax.set_xlabel("effect on log price per 1 SD of feature")
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as t:
        p = f"{t}/ridge_coefficients.png"; fig.savefig(p, dpi=110, bbox_inches="tight")
        mlflow.log_artifact(p, artifact_path="plots")
    plt.show(); plt.close()

    mlflow.sklearn.log_model(ridge, name="model")
    RIDGE_RUN_ID = run.info.run_id

print_metrics("RUN 2 — RIDGE BASELINE", m_ridge)
print(f"\n  Lift over city-median floor: {m_ridge['test_r2'] - BASELINE_R2:+.4f} R2")
print(f"  Fit time: {fit_s:.1f}s")

**Reading the Ridge result (test R² ≈ 0.626).**

Ridge more than doubles the city-median floor, so the engineered features carry
real signal. But the residual diagnostics show the mis-specification we
predicted in Notebook 01 §2.6: the *Residuals vs Fitted* panel is not a
structureless band. There is visible curvature and the residual spread widens at
both ends.

That is the Simpson's-paradox problem made visible. Ridge is forced to fit a
single global coefficient for `dist_center_km` when the true coefficient is
−0.32 in New York and +0.20 in Rio. It compromises on something near zero and
absorbs the error into the residuals. No amount of regularisation fixes a
model that cannot express the interaction — we need a hypothesis class that
learns interactions natively.

## 6. Run 3 — Random Forest (tuned, nested MLflow runs)

**Why Random Forest.** It is the natural first escalation from a linear model:

- **Learns interactions without specification.** Each tree splits on `city`,
  then on `dist_center_km` *within* that branch — recovering the city-specific
  geography of §2.6 automatically. This is precisely the capability Ridge lacks.
- **Handles our sentinel encoding correctly.** A split at `review_scores_rating_f
  > -0.5` cleanly separates the 32.7% never-reviewed population. A linear model
  would treat −1 as "one point below zero" and produce nonsense.
- **Bagging gives low-variance estimates** and out-of-the-box importances.
- **It is the standard tabular benchmark**, so failing to beat it would be a
  meaningful negative result.

We tune with **`GroupKFold` on `host_id`**, not plain `KFold`. Cross-validating
without groups reintroduces exactly the leakage the train/test split was built
to prevent, and would select hyperparameters that reward memorisation — deeper
trees, smaller leaves — precisely the wrong direction.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_grid = [
    {"n_estimators": 200, "max_depth": 16, "min_samples_leaf": 8,  "max_features": 0.4},
    {"n_estimators": 200, "max_depth": 24, "min_samples_leaf": 4,  "max_features": 0.4},
    {"n_estimators": 300, "max_depth": 24, "min_samples_leaf": 2,  "max_features": 0.5},
    {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 4, "max_features": 0.3},
]

gkf = GroupKFold(n_splits=4)
groups_train = meta_train["host_id"].values

best_rf, best_rf_cv, best_rf_params = None, -np.inf, None

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_rf_tuning_parent") as parent:
    mlflow.set_tags({**COMMON_TAGS, "model_family": "random_forest",
                     "run_role": "hyperparameter_search"})
    mlflow.log_params({"n_candidates": len(rf_grid), "cv_folds": 4,
                       "cv_strategy": "GroupKFold(host_id)",
                       "search_metric": "r2"})

    for i, params in enumerate(rf_grid, 1):
        with mlflow.start_run(run_name=f"rf_candidate_{i:02d}", nested=True):
            full = {**params, "random_state": RANDOM_STATE, "n_jobs": -1}
            mlflow.set_tags({**COMMON_TAGS, "model_family": "random_forest"})
            mlflow.log_params({**full, "model_type": "RandomForestRegressor"})

            pipe = Pipeline([("pre", make_preprocessor()),
                             ("model", RandomForestRegressor(**full))])
            t0 = time.time()
            cv = cross_val_score(pipe, X_train, y_train, groups=groups_train,
                                 cv=gkf, scoring="r2", n_jobs=1)
            pipe.fit(X_train, y_train)
            m = evaluate(pipe, X_train, y_train, X_test, y_test)
            mlflow.log_metrics({**m,
                                "cv_r2_mean": round(float(cv.mean()), 4),
                                "cv_r2_std":  round(float(cv.std()), 4),
                                "fit_seconds": round(time.time() - t0, 1)})
            mlflow.sklearn.log_model(pipe, name="model")

            print(f"  cand {i}: depth={params['max_depth']} "
                  f"leaf={params['min_samples_leaf']} "
                  f"| CV R2={cv.mean():.4f}+-{cv.std():.4f} "
                  f"| test R2={m['test_r2']:.4f} | gap={m['overfit_gap_r2']:+.3f}")

            if cv.mean() > best_rf_cv:
                best_rf_cv, best_rf, best_rf_params = cv.mean(), pipe, full

    mlflow.log_metric("best_cv_r2", round(float(best_rf_cv), 4))

print(f"\nBest RF by grouped CV: {best_rf_params}")
print(f"Best CV R2: {best_rf_cv:.4f}")

In [ ]:
with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_random_forest_best") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_family": "random_forest",
                     "run_role": "tuned_champion_candidate"})
    mlflow.log_params({**best_rf_params, "model_type": "RandomForestRegressor",
                       "selected_by": "GroupKFold CV R2"})

    m_rf = evaluate(best_rf, X_train, y_train, X_test, y_test)
    mlflow.log_metrics({**m_rf, "cv_r2_mean": round(float(best_rf_cv), 4),
                        "lift_over_ridge_r2": round(m_rf["test_r2"] - m_ridge["test_r2"], 4)})
    log_residual_plots(y_test, best_rf.predict(X_test), "Random Forest")
    rf_imp = log_importance(best_rf, feature_names(best_rf.named_steps["pre"]),
                            "Random Forest")
    mlflow.sklearn.log_model(best_rf, name="model")
    RF_RUN_ID = run.info.run_id

print_metrics("RUN 3 — RANDOM FOREST (tuned)", m_rf)
print(f"\n  Lift over Ridge : {m_rf['test_r2'] - m_ridge['test_r2']:+.4f} R2")

**Reading the Random Forest result (test R² ≈ 0.682).**

A substantial gain over Ridge — the interaction hypothesis was correct. But look
at the **overfit gap**: train R² ≈ 0.88 against test ≈ 0.68, a gap of ~0.20. The
forest is memorising far more than it generalises.

This is not fatal (bagged ensembles tolerate high individual-tree variance) but
it is diagnostic. Deep unpruned trees have enough capacity to isolate individual
host portfolios, which is exactly the behaviour the grouped split penalises. It
predicts what Notebook 01 §5.2 measured: Random Forest showed the **largest
leakage optimism of the three models (+0.046 R²)** when evaluated on a random
split. Capacity that is spent memorising hosts looks like performance under a
random split and evaporates under a grouped one.

That points us toward a model with the same interaction-learning ability but
better-controlled variance.

## 7. Run 4 — Histogram Gradient Boosting (tuned, nested MLflow runs)

**Why `HistGradientBoostingRegressor`.** Four reasons, and the fourth is an
MLOps reason rather than a modelling one:

1. **Boosting corrects residuals sequentially**, so each tree fixes what the
   ensemble still gets wrong. On heteroscedastic data — and §6's residual plot
   confirms ours is — this typically beats bagging.
2. **Explicit regularisation.** `l2_regularization`, `max_leaf_nodes` and
   `min_samples_leaf` control capacity directly, which is exactly the lever
   Random Forest lacked.
3. **Histogram binning makes it fast on 220k rows** — roughly 4× quicker than
   the equivalent Random Forest in our runs, which matters when you are running
   a grid.
4. **It ships inside scikit-learn.** LightGBM or XGBoost would likely perform
   comparably, but they would require a custom container or a `requirements.txt`
   in the SageMaker training job. `HistGradientBoostingRegressor` runs in the
   stock `sklearn 1.2-1` container with **zero** extra dependencies. Choosing a
   model that deploys without a bespoke image is a legitimate engineering
   criterion, and here it costs us nothing in accuracy.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_grid = [
    {"max_iter": 400, "learning_rate": 0.08, "max_leaf_nodes": 31,
     "min_samples_leaf": 40, "l2_regularization": 1.0},
    {"max_iter": 600, "learning_rate": 0.06, "max_leaf_nodes": 63,
     "min_samples_leaf": 40, "l2_regularization": 1.0},
    {"max_iter": 600, "learning_rate": 0.06, "max_leaf_nodes": 127,
     "min_samples_leaf": 20, "l2_regularization": 2.0},
    {"max_iter": 900, "learning_rate": 0.04, "max_leaf_nodes": 63,
     "min_samples_leaf": 60, "l2_regularization": 5.0},
]

best_hgb, best_hgb_cv, best_hgb_params = None, -np.inf, None

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_hgb_tuning_parent") as parent:
    mlflow.set_tags({**COMMON_TAGS, "model_family": "gradient_boosting",
                     "run_role": "hyperparameter_search"})
    mlflow.log_params({"n_candidates": len(hgb_grid), "cv_folds": 4,
                       "cv_strategy": "GroupKFold(host_id)", "search_metric": "r2"})

    for i, params in enumerate(hgb_grid, 1):
        with mlflow.start_run(run_name=f"hgb_candidate_{i:02d}", nested=True):
            full = {**params, "random_state": RANDOM_STATE,
                    "early_stopping": False}
            mlflow.set_tags({**COMMON_TAGS, "model_family": "gradient_boosting"})
            mlflow.log_params({**full, "model_type": "HistGradientBoostingRegressor"})

            pipe = Pipeline([("pre", make_preprocessor()),
                             ("model", HistGradientBoostingRegressor(**full))])
            t0 = time.time()
            cv = cross_val_score(pipe, X_train, y_train, groups=groups_train,
                                 cv=gkf, scoring="r2", n_jobs=1)
            pipe.fit(X_train, y_train)
            m = evaluate(pipe, X_train, y_train, X_test, y_test)
            mlflow.log_metrics({**m,
                                "cv_r2_mean": round(float(cv.mean()), 4),
                                "cv_r2_std":  round(float(cv.std()), 4),
                                "fit_seconds": round(time.time() - t0, 1)})
            mlflow.sklearn.log_model(pipe, name="model")

            print(f"  cand {i}: lr={params['learning_rate']} "
                  f"leaves={params['max_leaf_nodes']} iter={params['max_iter']} "
                  f"| CV R2={cv.mean():.4f}+-{cv.std():.4f} "
                  f"| test R2={m['test_r2']:.4f} | gap={m['overfit_gap_r2']:+.3f}")

            if cv.mean() > best_hgb_cv:
                best_hgb_cv, best_hgb, best_hgb_params = cv.mean(), pipe, full

    mlflow.log_metric("best_cv_r2", round(float(best_hgb_cv), 4))

print(f"\nBest HGB by grouped CV: {best_hgb_params}")
print(f"Best CV R2: {best_hgb_cv:.4f}")

In [ ]:
with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_hgb_best") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_family": "gradient_boosting",
                     "run_role": "tuned_champion_candidate"})
    mlflow.log_params({**best_hgb_params,
                       "model_type": "HistGradientBoostingRegressor",
                       "selected_by": "GroupKFold CV R2",
                       "container_compatibility": "sklearn 1.2-1 stock (no custom image)"})

    m_hgb = evaluate(best_hgb, X_train, y_train, X_test, y_test)
    mlflow.log_metrics({**m_hgb, "cv_r2_mean": round(float(best_hgb_cv), 4),
                        "lift_over_ridge_r2": round(m_hgb["test_r2"] - m_ridge["test_r2"], 4),
                        "lift_over_rf_r2":    round(m_hgb["test_r2"] - m_rf["test_r2"], 4)})
    log_residual_plots(y_test, best_hgb.predict(X_test), "HistGradientBoosting")
    mlflow.sklearn.log_model(best_hgb, name="model")
    HGB_RUN_ID = run.info.run_id

print_metrics("RUN 4 — HIST GRADIENT BOOSTING (tuned)", m_hgb)
print(f"\n  Lift over Ridge : {m_hgb['test_r2'] - m_ridge['test_r2']:+.4f} R2")
print(f"  Lift over RF    : {m_hgb['test_r2'] - m_rf['test_r2']:+.4f} R2")
print(f"  Overfit gap     : {m_hgb['overfit_gap_r2']:+.4f} "
      f"(vs RF {m_rf['overfit_gap_r2']:+.4f})")

**Reading the HGB result (test R² ≈ 0.712).**

The best of the three, and — more importantly — the best *for the right reason*.
Its overfit gap is roughly **0.07** against Random Forest's **0.20**. It reaches
higher test performance while memorising far less, which is exactly what the
explicit regularisation was for.

That difference is not cosmetic. A model with a small train/test gap degrades
gracefully as the data drifts; one with a large gap is relying on structure that
may not persist. For a model that will sit behind an endpoint for months, the
generalisation gap is as important as the headline metric.

## 8. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "City median (floor)": m_base,
    "Ridge":               m_ridge,
    "Random Forest":       m_rf,
    "HistGradientBoosting": m_hgb,
}).T[["test_r2", "test_rmse_log", "test_mdape", "test_within_25pct",
      "test_mae_usd", "train_r2", "overfit_gap_r2"]]
comparison.columns = ["Test R2", "RMSE(log)", "MdAPE", "within-25%",
                      "MAE USD", "Train R2", "Overfit gap"]

print("MODEL COMPARISON — host-grouped test set")
print("=" * 92)
print(comparison.round(4).to_string())
print("=" * 92)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
mods = comparison.index.tolist()
cols = ["#95A5A6", "#5B9BD5", "#E67E22", "#27AE60"]

axes[0].bar(mods, comparison["Test R2"], color=cols)
axes[0].axhline(QUALITY_GATE_R2, ls="--", c="red", label=f"gate {QUALITY_GATE_R2}")
axes[0].axhline(contract["estimated_r2_ceiling"], ls=":", c="black",
                label=f"est. ceiling {contract['estimated_r2_ceiling']:.2f}")
axes[0].set_ylabel("Test R2"); axes[0].set_title("Explained variance", fontweight="bold")
axes[0].legend(fontsize=8); axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(mods, comparison["MdAPE"] * 100, color=cols)
axes[1].set_ylabel("Median APE (%)")
axes[1].set_title("Typical host-facing error", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(mods, comparison["Train R2"], color=cols, alpha=.45, label="train")
axes[2].bar(mods, comparison["Test R2"], color=cols, label="test")
axes[2].set_title("Train vs test — generalisation gap", fontweight="bold")
axes[2].legend(fontsize=8); axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

**What the comparison says.**

| Model | Test R² | MdAPE | Overfit gap | Verdict |
|---|---|---|---|---|
| City median | ~0.44 | ~40% | 0.00 | The floor. Free, and not useless. |
| Ridge | 0.626 | 29.8% | +0.01 | Interpretable, structurally mis-specified |
| Random Forest | 0.682 | 27.3% | +0.20 | Strong, but memorises hosts |
| **HistGradientBoosting** | **0.712** | **25.8%** | **+0.07** | **Champion** |

Three observations worth stating explicitly:

1. **Diminishing returns are real.** Ridge captures most of the achievable gain
   over the floor; the two tree ensembles add roughly 0.09 R² on top of it. If
   interpretability were a hard regulatory requirement, shipping Ridge would be
   a defensible engineering decision, not a failure.
2. **The ranking would have been different under a random split.** RF and HGB
   score 0.728 and 0.739 respectively on a random split — nearly tied. Under the
   honest grouped split they separate by 0.03. The evaluation protocol changed
   the decision, which is the entire argument for taking splits seriously.
3. **We are at ~80% of the estimated ceiling** (0.712 / 0.89). The remaining gap
   is mostly the host-discretion noise quantified in Notebook 01 §5.3, not
   model capacity we failed to use.

## 9. Deep Error Analysis

A single test R² tells you almost nothing about whether a model is safe to
deploy. This section asks the questions that actually matter: *where* does the
model fail, *who* does it fail for, and *is the failure systematic*.

We analyse the champion (HGB) on the held-out grouped test set throughout.

In [ ]:
CHAMPION = best_hgb
CHAMPION_NAME = "HistGradientBoosting"

pred_log = CHAMPION.predict(X_test)
err = meta_test.copy()
err["actual_log"] = y_test.values
err["pred_log"]   = pred_log
err["residual"]   = err.actual_log - err.pred_log
err["actual_usd"] = np.exp(err.actual_log)
err["pred_usd"]   = np.exp(err.pred_log)
err["ape"]        = np.abs(err.pred_usd - err.actual_usd) / err.actual_usd
err["signed_pct"] = (err.pred_usd - err.actual_usd) / err.actual_usd

print(f"Error frame built for {len(err):,} held-out listings.")
print(f"Overall: R2={r2_score(err.actual_log, err.pred_log):.4f}  "
      f"MdAPE={err.ape.median():.1%}  bias={err.residual.mean():+.4f}")

### 9.1 Regression to the mean — the dominant failure mode

The first thing to check in any regression is whether error is uniform across
the target range. It almost never is.

In [ ]:
err["decile"] = pd.qcut(err.actual_usd, 10, labels=False)
dec = err.groupby("decile").agg(
    n=("residual", "size"), price_lo=("actual_usd", "min"),
    price_hi=("actual_usd", "max"), bias_log=("residual", "mean"),
    mdape=("ape", "median"), signed_pct=("signed_pct", "median"))
dec["mean_pred_usd"]   = err.groupby("decile").pred_usd.median()
dec["mean_actual_usd"] = err.groupby("decile").actual_usd.median()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(dec.index, dec.bias_log,
            color=["#C0392B" if v < 0 else "#27AE60" for v in dec.bias_log])
axes[0].axhline(0, c="black", lw=.8)
axes[0].set_xlabel("true price decile"); axes[0].set_ylabel("mean residual (log)")
axes[0].set_title("Systematic bias by price level", fontweight="bold")

axes[1].plot(dec.index, dec.mean_actual_usd, "o-", label="actual", color="#2C3E50")
axes[1].plot(dec.index, dec.mean_pred_usd, "s--", label="predicted", color="#E67E22")
axes[1].set_yscale("log"); axes[1].set_xlabel("true price decile")
axes[1].set_ylabel("median USD (log scale)"); axes[1].legend()
axes[1].set_title("Predictions compress toward the middle", fontweight="bold")

axes[2].plot(dec.index, dec.mdape * 100, "o-", color="#8E44AD")
axes[2].set_xlabel("true price decile"); axes[2].set_ylabel("MdAPE (%)")
axes[2].set_title("Error is U-shaped in price", fontweight="bold")
plt.tight_layout(); plt.show()

print(dec.round(3).to_string())

**This is the single most important finding of the error analysis.**

The bias runs monotonically from **−0.40 log points in the cheapest decile to
+0.56 in the most expensive**. In plain terms:

- The model **over-prices the cheapest listings by roughly 40%**
- The model **under-prices the most expensive listings by roughly 50%**
- MdAPE is U-shaped: ~45% at the bottom, ~20% in the middle, ~39% at the top

This is **regression to the mean**, and it is not a bug — it is the mathematically
correct behaviour of a squared-error model under irreducible noise. When
features cannot fully determine the target, the conditional-mean estimate that
minimises MSE necessarily shrinks toward the centre of the distribution. Any
unbiased-at-the-extremes model would have *higher* total error.

But correct-in-expectation is not the same as safe-in-deployment. The business
consequence is severe and asymmetric:

> A budget host is told to raise their price by 40%, and loses bookings.
> A luxury host is told to cut their price by a third, and leaves money on the table.

Both are actively harmful recommendations delivered with the same confidence as
the accurate mid-market ones. **Mitigations are specified in Notebook 03**:
segment-conditional confidence intervals, refusal to recommend outside the
calibrated middle deciles, and quantile-regression alternatives for the tails.

### 9.2 Performance by city

In [ ]:
city_err = err.groupby("city").agg(
    n=("residual", "size"), bias=("residual", "mean"),
    resid_sd=("residual", "std"), mdape=("ape", "median"),
    median_price=("actual_usd", "median"))
city_err["r2"] = err.groupby("city").apply(
    lambda g: r2_score(g.actual_log, g.pred_log))
city_err = city_err.sort_values("mdape")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
axes[0].barh(city_err.index, city_err.mdape * 100,
             color=["#27AE60" if v < .27 else "#E67E22" if v < .32 else "#C0392B"
                    for v in city_err.mdape])
axes[0].axvline(err.ape.median() * 100, ls="--", c="black",
                label=f"overall {err.ape.median():.1%}")
axes[0].set_xlabel("MdAPE (%)"); axes[0].legend()
axes[0].set_title("Median error by city — a 1.6x spread", fontweight="bold")

axes[1].scatter(city_err.n, city_err.mdape * 100, s=90, color="#5B9BD5")
for c, r in city_err.iterrows():
    axes[1].annotate(c, (r.n, r.mdape * 100), fontsize=8,
                     xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("test listings in city"); axes[1].set_ylabel("MdAPE (%)")
axes[1].set_title("Error vs sample size", fontweight="bold")
plt.tight_layout(); plt.show()

print(city_err.round(3).to_string())

**Performance is far from uniform.** Paris achieves 21.2% MdAPE; Hong Kong
34.0%. That is a **1.6× spread in the quality of service** delivered to hosts
depending on which city they operate in.

Two distinct causes, and they call for different responses:

- **Sample size.** Hong Kong contributes the fewest listings (~1,200 in test) and
  has both the worst MdAPE and the lowest per-city R² (0.475). Data scarcity —
  addressable by collecting more data or by explicitly modelling city as a
  hierarchical effect.
- **Intrinsic market heterogeneity.** Rome has a respectable MdAPE (24.9%) but a
  low R² (0.507), meaning its prices are compressed — there is simply less
  variance to explain. Rio and Istanbul have high MdAPE *and* high residual SD:
  genuinely dispersed markets where identical-looking listings really do charge
  very different prices.

The **bias column is near zero everywhere** (|bias| ≤ 0.05), which is the
reassuring part: the model is not systematically over- or under-pricing any
whole city. The disparity is in *precision*, not *direction* — a fairness
concern about unequal service quality rather than systematic disadvantage. Both
are recorded in the model card.

### 9.3 Performance by room type and by host segment

In [ ]:
seg_room = err.groupby("room_type").agg(
    n=("residual", "size"), bias=("residual", "mean"),
    resid_sd=("residual", "std"), mdape=("ape", "median"))
seg_rev = err.groupby("has_reviews").agg(
    n=("residual", "size"), bias=("residual", "mean"), mdape=("ape", "median"))
seg_rev.index = ["never reviewed (cold start)", "has reviews"]
seg_host = err.groupby("is_professional_host").agg(
    n=("residual", "size"), bias=("residual", "mean"), mdape=("ape", "median"))
seg_host.index = ["individual host (<5 listings)", "professional host (>=5)"]

print("BY ROOM TYPE"); print(seg_room.round(3).to_string())
print("\nBY REVIEW STATUS"); print(seg_rev.round(3).to_string())
print("\nBY HOST TYPE"); print(seg_host.round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, d, title in [(axes[0], seg_room, "Room type"),
                     (axes[1], seg_rev,  "Review status"),
                     (axes[2], seg_host, "Host type")]:
    ax.barh(d.index, d.mdape * 100,
            color=["#C0392B" if v > .30 else "#5B9BD5" for v in d.mdape])
    ax.axvline(err.ape.median() * 100, ls="--", c="black")
    ax.set_xlabel("MdAPE (%)"); ax.set_title(title, fontweight="bold")
    for i, (lbl, r) in enumerate(d.iterrows()):
        ax.text(r.mdape * 100 + .4, i, f"n={int(r.n):,}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

**Three segment findings, in descending order of concern.**

**1. Shared rooms are near-unusable (MdAPE 44.0%, residual SD 0.72).** The worst
segment by a wide margin, on only ~900 test listings. Shared-room pricing is
driven by hostel-style dynamics — per-bed rates, dorm size, seasonal backpacker
demand — that our feature set does not capture at all. The model should
**decline to serve this segment** rather than emit a number with 44% typical
error.

**2. Cold-start listings are the worst-served (31.0% vs 23.9%).** Listings with
no reviews get error nearly a third higher than reviewed ones. This is a
**painful inversion of need**: a brand-new host has no market feedback, no
booking history, and no intuition — they are precisely the user for whom pricing
guidance is most valuable, and they receive the least reliable version of it.
This is a genuine fairness issue, not merely a technical one, and it is not
fixable by better modelling of the existing features — the information is not in
the data.

**3. Professional hosts get worse predictions (29.6% vs 24.8%).** Initially
counter-intuitive, since these are the sophisticated operators. The explanation
is that professionals price *strategically*: dynamic and seasonal rates,
portfolio-level yield management, deliberate loss-leaders. Individual hosts price
closer to a simple market comparison, which is exactly what the model computes.
The model is well-suited to the amateur host and poorly suited to the
professional — which happily matches the product intent, but should be stated
rather than assumed.

**Hotel rooms carry a −0.074 bias**: systematically over-priced. Hotel inventory
on Airbnb competes with the hotel channel, whose pricing logic is entirely
outside this dataset.

### 9.4 The worst micro-markets

In [ ]:
nb = (err.groupby(["city", "neighbourhood"])
        .agg(n=("ape", "size"), mdape=("ape", "median"),
             bias=("residual", "mean"), median_price=("actual_usd", "median"))
        .query("n >= 150"))

worst = nb.nlargest(12, "mdape")
best  = nb.nsmallest(8, "mdape")

print(f"Neighbourhoods with >=150 test listings: {len(nb)}")
print("\nWORST 12 by MdAPE"); print(worst.round(3).to_string())
print("\nBEST 8 by MdAPE");  print(best.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
lbl_w = [f"{c[:3]} | {n[:22]}" for c, n in worst.index]
lbl_b = [f"{c[:3]} | {n[:22]}" for c, n in best.index]
ax.barh(lbl_w[::-1], worst.mdape.values[::-1] * 100, color="#C0392B", label="worst")
ax.barh(lbl_b[::-1], best.mdape.values[::-1] * 100, color="#27AE60", label="best")
ax.axvline(err.ape.median() * 100, ls="--", c="black",
           label=f"overall {err.ape.median():.1%}")
ax.set_xlabel("MdAPE (%)"); ax.legend()
ax.set_title("Best and worst micro-markets (>=150 test listings)", fontweight="bold")
plt.tight_layout(); plt.show()

The spread across micro-markets is **~2.6×**: Rome's Appia Antica at 16.9%
against Bangkok's Bang Rak at 44.8%.

The pattern is not random. The best-served neighbourhoods are dense, homogeneous
European residential districts — Paris's Ménilmontant, Reuilly, Enclos-St-Laurent
— where listings are genuinely comparable to one another. The worst are
**tourist-core districts in emerging markets**: Bang Rak and Phra Nakhon in
Bangkok, Fatih in Istanbul, Yau Tsim Mong in Hong Kong. These neighbourhoods mix
$15 hostel beds with $300 boutique suites inside the same postal area, so
"comparable listings in your neighbourhood" is a much weaker signal there.

**Operational consequence.** This argues for a **per-neighbourhood confidence
tier** shipped alongside the point estimate, rather than a single global
disclaimer. A host in Reuilly should see a tight range; a host in Bang Rak should
see a wide one, or a recommendation that the model declines to make.

### 9.5 Worst individual predictions — qualitative inspection

In [ ]:
worst_rows = err.nlargest(12, "ape")[
    ["city", "neighbourhood", "room_type", "actual_usd", "pred_usd",
     "signed_pct", "has_reviews", "is_professional_host"]].copy()
worst_rows["actual_usd"] = worst_rows.actual_usd.round(0)
worst_rows["pred_usd"]   = worst_rows.pred_usd.round(0)
worst_rows["signed_pct"] = (worst_rows.signed_pct * 100).round(0)

print("12 WORST PREDICTIONS (by absolute percentage error)")
print(worst_rows.to_string(index=False))

over  = (err.signed_pct > 0).mean()
tail  = err.ape > err.ape.quantile(0.95)
print(f"\nDirectional balance: {over:.1%} over-predicted, {1-over:.1%} under")
print(f"\nComposition of the worst 5% of predictions vs the test set overall:")
print(pd.DataFrame({
    "worst 5%": [err[tail].has_reviews.mean(), err[tail].is_professional_host.mean(),
                 err[tail].actual_usd.median()],
    "overall":  [err.has_reviews.mean(), err.is_professional_host.mean(),
                 err.actual_usd.median()],
}, index=["share with reviews", "share professional host",
          "median actual USD"]).round(3).to_string())

Inspecting the individual failures confirms the aggregate story: the worst
predictions cluster on **very cheap listings in expensive neighbourhoods** and
**very expensive listings in cheap neighbourhoods** — precisely the cases where
the neighbourhood target encoding, our strongest single feature, actively
misleads. A $20 shared room in central Paris inherits the Paris-centre prior and
gets priced at $90.

This is the cost of the feature that buys us the most performance. It is a
reasonable trade, but it means **neighbourhood-atypical listings are a known,
characterised failure mode**, not a surprise — and it should appear in the model
card as such.

### 9.6 Permutation importance — what the model actually relies on

In [ ]:
from sklearn.inspection import permutation_importance

sub = np.random.RandomState(RANDOM_STATE).choice(len(X_test),
                                                 min(20000, len(X_test)),
                                                 replace=False)
with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_interpretability") as run:
    mlflow.set_tags({**COMMON_TAGS, "run_role": "interpretability_governance"})
    t0 = time.time()
    pi = permutation_importance(CHAMPION, X_test.iloc[sub], y_test.iloc[sub],
                                n_repeats=3, random_state=RANDOM_STATE,
                                scoring="r2", n_jobs=2)
    imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 8))
    imp.head(25)[::-1].plot(kind="barh", ax=ax, color="#5B9BD5")
    ax.set_xlabel("drop in test R2 when the feature is shuffled")
    ax.set_title(f"Permutation importance — {CHAMPION_NAME} (top 25)",
                 fontweight="bold")
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as t:
        p = f"{t}/permutation_importance.png"
        fig.savefig(p, dpi=110, bbox_inches="tight")
        mlflow.log_artifact(p, artifact_path="plots")
        imp.to_frame("importance").to_csv(f"{t}/permutation_importance.csv")
        mlflow.log_artifact(f"{t}/permutation_importance.csv", artifact_path="analysis")
    plt.show(); plt.close()

    mlflow.log_metrics({f"perm_imp_{k}": round(float(v), 5)
                        for k, v in imp.head(10).items()})
    mlflow.log_metric("perm_importance_seconds", round(time.time() - t0, 1))

print("Top 20 permutation importances (drop in R2):")
print(imp.head(20).round(4).to_string())

**We use permutation importance rather than impurity-based importance.** Split-count
importances are biased toward high-cardinality features and are computed on
*training* data, so they reward memorisation. Permutation importance measures the
drop in *test* R² when a feature is shuffled — it answers "what does this model
actually need to generalise?"

**Location dominates and it is not close.** `city_neigh` costs **0.304 R²** when
shuffled and `city` a further **0.128**; together, geography is roughly 60% of
the model's explanatory power. Physical capacity (`accommodates` 0.103,
`room_type` 0.089, `bedrooms` 0.065) is the second block. Everything else is
marginal.

**Three checks that the model is behaving sensibly:**

- **The negative control held.** Notebook 01 §2.8 predicted review scores would
  be near-useless; `review_scores_value_f` ranks 18th at 0.0056 — two orders of
  magnitude below `city_neigh`. The model did not manufacture importance where
  the EDA said there was none, which is genuine evidence against overfitting.
- **`log_host_listings` at 0.0248 is the highest-ranked host feature.** The
  model has learned that professional operators price differently — the
  city-dependent effect from Notebook 01 §4.3.
- **`host_acceptance_rate_f` at 0.0247 is suspiciously high** for a variable that
  is 40% missing. Most likely the model is using *missingness* as a maturity
  proxy rather than the rate itself. Flagged for monitoring: if Airbnb changed
  its suppression rules, this feature's meaning would shift silently.

**Governance implication.** A model that is 60% geography is, functionally, a
location-price index. That is defensible for pricing guidance — location genuinely
is what drives accommodation prices — but it means the model will **faithfully
reproduce whatever spatial price structure exists in the training data**,
including any structure that reflects historical segregation or discriminatory
pricing. Notebook 03 tests this directly.

## 10. Quality Gate and Champion Selection

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.test_r2 DESC"])

cols = ["tags.mlflow.runName", "metrics.test_r2", "metrics.test_mdape",
        "metrics.test_rmse_log", "metrics.test_within_25pct",
        "metrics.overfit_gap_r2", "run_id"]
avail = [c for c in cols if c in runs.columns]
print("ALL RUNS RANKED BY TEST R2")
print(runs[avail].head(10).to_string(index=False))

best_run     = runs.iloc[0]
BEST_RUN_ID  = best_run["run_id"]
BEST_R2      = float(best_run["metrics.test_r2"])
BEST_NAME    = best_run.get("tags.mlflow.runName", "unknown")
BEST_MDAPE   = float(best_run.get("metrics.test_mdape", np.nan))

In [ ]:
print("=" * 66)
print("QUALITY GATE")
print("=" * 66)
print(f"  Champion run   : {BEST_NAME}")
print(f"  Test R2        : {BEST_R2:.4f}")
print(f"  Required       : {QUALITY_GATE_R2:.2f}")
print(f"  Est. ceiling   : {contract['estimated_r2_ceiling']:.4f}")
print(f"  % of ceiling   : {BEST_R2/contract['estimated_r2_ceiling']:.1%}")
print("-" * 66)

if BEST_R2 < QUALITY_GATE_R2:
    print("  RESULT: FAILED — do not promote.")
    print("  Revisit feature engineering or the hyperparameter search space.")
    BEST_MODEL_URI = None
else:
    print("  RESULT: PASSED — cleared for the production pipeline.")
    BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"
    print(f"  Model URI      : {BEST_MODEL_URI}")

print("=" * 66)
print("\nGate caveats recorded for Notebook 03 (a passing R2 is necessary,")
print("not sufficient):")
print("  - Shared rooms:   MdAPE ~44%  -> segment should be excluded from serving")
print("  - Cold start:     MdAPE ~31%  -> widen intervals / flag low confidence")
print("  - Price extremes: +-40-50% bias in deciles 1 and 10")
print("  - Hong Kong:      MdAPE ~34%  -> lowest-confidence market")

In [ ]:
# ---------------------------------------------------------------------------
# Hand-off record for Notebook 03. The known-limitations block travels WITH
# the model so the deployment notebook cannot silently ignore it.
# ---------------------------------------------------------------------------
best_model_info = {
    "team_id": TEAM_ID, "student_id": STUDENT_ID,
    "project_name": PROJECT_NAME, "task_type": "regression",
    "target": TARGET, "target_transform": "natural log",
    "best_run_id": BEST_RUN_ID, "best_run_name": BEST_NAME,
    "best_test_r2": round(BEST_R2, 4),
    "best_test_mdape": round(BEST_MDAPE, 4),
    "best_model_uri": BEST_MODEL_URI,
    "quality_gate_r2": QUALITY_GATE_R2,
    "estimated_r2_ceiling": contract["estimated_r2_ceiling"],
    "champion_family": "HistGradientBoostingRegressor",
    "champion_hyperparameters": best_hgb_params,
    "mlflow_tracking_uri": MLFLOW_APP_ARN,
    "mlflow_experiment": MLFLOW_EXPERIMENT,
    "tracking_backend": TRACKING_BACKEND,
    "mlflow_app_team_tag": TEAM_ID,
    "model_comparison": {
        "city_median_floor": m_base["test_r2"],
        "ridge":             m_ridge["test_r2"],
        "random_forest":     m_rf["test_r2"],
        "hist_gradient_boosting": m_hgb["test_r2"],
    },
    "known_limitations": {
        "regression_to_mean": {
            "decile_1_bias_log": float(dec.bias_log.iloc[0]),
            "decile_10_bias_log": float(dec.bias_log.iloc[-1]),
            "note": "over-prices cheap listings, under-prices expensive ones"},
        "worst_segment": {"name": "Shared room",
                          "mdape": float(seg_room.loc["Shared room", "mdape"])
                          if "Shared room" in seg_room.index else None,
                          "action": "exclude from serving"},
        "cold_start": {"mdape_no_reviews": float(seg_rev.mdape.iloc[0]),
                       "mdape_with_reviews": float(seg_rev.mdape.iloc[1]),
                       "action": "widen confidence interval"},
        "worst_city": {"name": city_err.index[-1],
                       "mdape": float(city_err.mdape.iloc[-1])},
        "best_city":  {"name": city_err.index[0],
                       "mdape": float(city_err.mdape.iloc[0])},
        "geography_share_of_importance": round(
            float(imp.get("city_neigh", 0) + imp.get("city", 0)) / float(imp.sum()), 3),
    },
}

Path("best_model.json").write_text(json.dumps(best_model_info, indent=2))
(LOCAL_ARTIFACTS / "best_model.json").write_text(json.dumps(best_model_info, indent=2))
if AWS_AVAILABLE:
    s3.put_object(Bucket=BUCKET, Key=f"{pp}/best_model.json",
                  Body=json.dumps(best_model_info, indent=2))

print("Saved best_model.json")
print(json.dumps(best_model_info, indent=2)[:1600] + "\n  ...")

### 10.1 Verify the logged model round-trips

In [ ]:
# Load the champion back from MLflow exactly as a downstream consumer would.
# A model that logs but does not reload is worse than no model.
loaded = mlflow.sklearn.load_model(BEST_MODEL_URI)

sample_X = X_test.head(8)
sample_m = meta_test.head(8)
p_log = loaded.predict(sample_X)

check = pd.DataFrame({
    "city": sample_m.city.values,
    "room_type": sample_m.room_type.values,
    "actual_usd": np.exp(y_test.head(8)).round(0).values,
    "predicted_usd": np.exp(p_log).round(0),
})
check["error_pct"] = ((check.predicted_usd - check.actual_usd)
                      / check.actual_usd * 100).round(1)

print("Round-trip inference from the MLflow model URI:")
print(check.to_string(index=False))

orig = CHAMPION.predict(sample_X)
assert np.allclose(orig, p_log, atol=1e-8), "Reloaded model disagrees with in-memory!"
print("\n[OK] Reloaded model reproduces in-memory predictions exactly.")

In [ ]:
print("=" * 66)
print("NOTEBOOK 02 COMPLETE")
print("=" * 66)
print(f"Experiment      : {MLFLOW_EXPERIMENT}")
print(f"Tracking        : {TRACKING_BACKEND}")
print(f"Runs logged     : baseline + ridge + {len(rf_grid)} RF + {len(hgb_grid)} HGB")
print(f"                  + 2 tuning parents + 2 champions + interpretability")
print(f"Champion        : {BEST_NAME}")
print(f"  Test R2       : {BEST_R2:.4f}  (gate {QUALITY_GATE_R2})")
print(f"  Test MdAPE    : {BEST_MDAPE:.1%}")
print(f"  Model URI     : {BEST_MODEL_URI}")
print()
print("Next: Notebook 03 — wrap this pipeline into a reusable production")
print("      artefact and orchestrate it with SageMaker Pipelines.")

---

## Checklist before Notebook 03

- [ ] MLflow App `TeamId` tag validated before any logging
- [ ] Feature contract loaded from Notebook 01 — no re-declared column lists
- [ ] Host overlap between train and test asserted to be 0
- [ ] Target encoder is out-of-fold; leakage prevented structurally
- [ ] Two-tier baseline established (city median, then Ridge)
- [ ] Two advanced models tuned with `GroupKFold`, all candidates as nested runs
- [ ] Every run logs the identical metric set in both log and USD space
- [ ] Residual diagnostics, importances and analysis CSVs logged as artefacts
- [ ] Regression-to-the-mean quantified by decile
- [ ] Segment analysis across city / room type / review status / host type
- [ ] Worst micro-markets and worst individual predictions inspected
- [ ] Permutation importance computed on test data, negative control verified
- [ ] Quality gate applied; `best_model.json` written **with known limitations**
- [ ] Champion reloaded from its MLflow URI and verified to reproduce predictions

### Results summary

| Model | Test R² | MdAPE | within-25% | Overfit gap |
|---|---|---|---|---|
| City median (floor) | ~0.44 | ~40% | — | 0.00 |
| Ridge | 0.626 | 29.8% | 42.0% | +0.007 |
| Random Forest | 0.682 | 27.3% | 46.5% | +0.198 |
| **HistGradientBoosting** | **0.712** | **25.8%** | **48.6%** | **+0.070** |

### Error-analysis findings carried into deployment

| Finding | Deployment action |
|---|---|
| Decile-1 bias −0.40, decile-10 bias +0.56 | Segment-conditional intervals; no point estimate at the extremes |
| Shared rooms MdAPE 44% | Exclude the segment from serving |
| Cold-start MdAPE 31% vs 24% | Low-confidence flag on never-reviewed listings |
| Hong Kong MdAPE 34% vs Paris 21% | Per-city confidence tiers |
| Geography ≈ 60% of importance | Spatial-bias audit before launch (Notebook 03) |